In [ ]:
!pip install rdkit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/CompDReAM/training_dataset.csv", low_memory=False)
mask = (df["Molecule ChEMBL ID"] == "CHEMBL153479") & (df["ChEMBLTargetID"] == "CHEMBL2179")
idx_to_remove = df[mask].index[0]  # assuming there's only one match
print("Index to remove:", idx_to_remove)
df_filtered = df.drop(index=idx_to_remove).reset_index(drop=True)
print("Dataset shape:", df_filtered.shape)
df_filtered.head()

import numpy as np
X_all = np.load("/content/drive/MyDrive/CompDReAM/v5/X_all.npy")
y = np.load("/content/drive/MyDrive/CompDReAM/v5/y.npy")
print("✓ X_all shape:", X_all.shape)
X_mol = X_all[:, :2048]           # Morgan fingerprints (2048 bits)
X_prot = X_all[:, 2048:3072]      # ProtBERT embeddings (1024 dims)
X_desc = X_all[:, 3072:]          # RDKit descriptors (217 dims)
print("✓ X_mol shape:", X_mol.shape)
print("✓ X_prot shape:", X_prot.shape)
print("✓ X_desc shape:", X_desc.shape)
print("✓ y shape:", y.shape)

X_mol = np.delete(X_mol, idx_to_remove, axis=0)
X_prot = np.delete(X_prot, idx_to_remove, axis=0)
X_desc = np.delete(X_desc, idx_to_remove, axis=0)
y = np.delete(y, idx_to_remove)
print("✓ X_mol shape:", X_mol.shape)
print("✓ X_prot shape:", X_prot.shape)
print("✓ X_desc shape:", X_desc.shape)
print("✓ y shape:", y.shape)

X_all = np.concatenate([X_mol, X_prot, X_desc], axis=1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Index to remove: 13812
Dataset shape: (116751, 30)
✓ X_all shape: (116752, 3289)
✓ X_mol shape: (116752, 2048)
✓ X_prot shape: (116752, 1024)
✓ X_desc shape: (116752, 217)
✓ y shape: (116752,)
✓ X_mol shape: (116751, 2048)
✓ X_prot shape: (116751, 1024)
✓ X_desc shape: (116751, 217)
✓ y shape: (116751,)


In [ ]:
# Set df_train to filtered df with pair and bin columns for splitting
df_train = df_filtered.copy()
df_train["UniProt_List"] = df_train["UniProt"].str.split(",").apply(lambda lst: [i.strip() for i in lst])
df_train["BestUniProt"] = df_train["UniProt_List"].apply(lambda lst: lst[0])
df_train["pair"] = df_train["Canonical SMILES"] + "_" + df_train["BestUniProt"]
df_train["bin"] = pd.qcut(df_train["pChEMBL"], q=5, labels=False, duplicates="drop")

# For splitting
unique_pairs = df_train.drop_duplicates(subset="pair")
pair_bins = unique_pairs.set_index("pair")["bin"].to_dict()
df_train["pair_bin"] = df_train["pair"].map(pair_bins)

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Train/test split on unique pairs
train_pairs, test_pairs = train_test_split(
    list(pair_bins.keys()),
    test_size=0.2,
    stratify=list(pair_bins.values()),
    random_state=42
)
train_idx = df_train[df_train["pair"].isin(train_pairs)].index
test_idx = df_train[df_train["pair"].isin(test_pairs)].index

X_train_split = X_all[train_idx]
X_test_split = X_all[test_idx]
y_train_split = y[train_idx]
y_test_split = y[test_idx]

print(f"Train size: {X_train_split.shape}, Test size: {X_test_split.shape}")

Train size: (93523, 3289), Test size: (23228, 3289)


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from math import sqrt

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_split, y_train_split)

y_pred_test = model.predict(X_test_split)
r2 = r2_score(y_test_split, y_pred_test)
rmse = sqrt(mean_squared_error(y_test_split, y_pred_test))

print(f"→ Test R²: {r2:.3f}")
print(f"→ Test RMSE: {rmse:.3f}")

→ Test R²: 0.724
→ Test RMSE: 0.523


In [5]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.DataStructs.cDataStructs import ConvertToNumpyArray
import numpy as np

from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
generator = GetMorganGenerator(radius=2, fpSize=2048)

# === Input: Ambroxol and GBA1 ===
smiles = "Nc1c(Br)cc(Br)cc1CN[C@H]1CC[C@H](O)CC1"  # Ambroxol
uniprot_id = "B7Z6S9"  # Or "P04062", "A0A068F658"
real_pchembl = 5.39

# === Morgan fingerprint ===
mol = Chem.MolFromSmiles(smiles)
fp = generator.GetFingerprint(mol)
fp_arr = np.zeros((1,), dtype=int)
ConvertToNumpyArray(fp, fp_arr)
X_mol = fp_arr.reshape(1, -1)

# === ProtBERT embedding (if UniProt present in training) ===
try:
    prot_index = df_train["BestUniProt"].tolist().index(uniprot_id)
    X_prot_query = X_prot[prot_index].reshape(1, -1)
except ValueError:
    raise ValueError(f"UniProt ID {uniprot_id} not found in training data. You may need to re-embed it manually.")

# === RDKit descriptors ===
desc_names = [desc[0] for desc in Descriptors.descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
X_desc = np.array(calc.CalcDescriptors(mol)).reshape(1, -1)
X_desc = np.nan_to_num(X_desc, nan=0.0, posinf=1e6, neginf=-1e6)
X_desc = np.clip(X_desc, -1e6, 1e6)

# === Combine features and predict ===
X_query = np.concatenate([X_mol, X_prot_query, X_desc], axis=1).astype(np.float32)
pred_pchembl = model.predict(X_query)[0]

# === Report
print(f"✓ Predicted pChEMBL for Ambroxol–GBA1: {pred_pchembl:.3f}")
print(f"✓ Actual pChEMBL: {real_pchembl:.3f}")
print(f"→ Δ = {abs(pred_pchembl - real_pchembl):.3f}")

✓ Predicted pChEMBL for Ambroxol–GBA1: 5.002
✓ Actual pChEMBL: 5.390
→ Δ = 0.388


In [6]:
import pandas as pd

results = {
    "SMILES": smiles,
    "UniProt": uniprot_id,
    "Predicted pChEMBL": pred_pchembl,
    "Actual pChEMBL": real_pchembl,
    "Delta": abs(pred_pchembl - real_pchembl)
}

pd.DataFrame([results]).to_csv("/content/drive/MyDrive/CompDReAM/ambroxol_gba1_result.csv", index=False)